In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans

# Load dataset
df = pd.read_csv("india_election_2019.csv")  # use your exact file name
df.head()

,Unnamed: 0,#,PC Name,No,Type,State,Winning Candidate,Party,Electors,Votes,Turnout,Margin,Margin %
0,0,543,Vellore,8,GEN,Tamil Nadu,Kathir Anand,Dravida Munetra Kazhagam,1438643.0,1028100.0,71.46,8141.0,0.79
1,1,542,Karimganj,1,SC,Assam,Kripanath Mallah,Bharatiya Janta Party,1165997.0,1061160.0,79.18,38389.0,3.62
2,2,1,Adilabad,1,ST,Telangana,Soyam Bapu Rao,Bharatiya Janta Party,1382837.0,1063730.0,77.90,58560.0,5.50
3,3,2,Agra,18,SC,Uttar Pradesh,Satyapal Singh Baghel,Bharatiya Janta Party,1866262.0,1145323.0,61.70,211546.0,18.50
4,4,3,Ahmadnagar,37,GEN,Maharashtra,Dr. Sujay Radhakrishna Vikhepatil,Bharatiya Janta Party,1793677.0,1203797.0,67.30,281474.0,23.40


CLEAN COLUMN NAMES

In [3]:
df.columns = df.columns.str.lower().str.replace(" ", "_").str.replace(".", "")
df.head()

,unnamed:_0,#,pc_name,no,type,state,winning_candidate,party,electors,votes,turnout,margin,margin_%
0,0,543,Vellore,8,GEN,Tamil Nadu,Kathir Anand,Dravida Munetra Kazhagam,1438643.0,1028100.0,71.46,8141.0,0.79
1,1,542,Karimganj,1,SC,Assam,Kripanath Mallah,Bharatiya Janta Party,1165997.0,1061160.0,79.18,38389.0,3.62
2,2,1,Adilabad,1,ST,Telangana,Soyam Bapu Rao,Bharatiya Janta Party,1382837.0,1063730.0,77.90,58560.0,5.50
3,3,2,Agra,18,SC,Uttar Pradesh,Satyapal Singh Baghel,Bharatiya Janta Party,1866262.0,1145323.0,61.70,211546.0,18.50
4,4,3,Ahmadnagar,37,GEN,Maharashtra,Dr. Sujay Radhakrishna Vikhepatil,Bharatiya Janta Party,1793677.0,1203797.0,67.30,281474.0,23.40


In [4]:
df = df.drop(columns=["unnamed_0"], errors="ignore")

In [5]:
df = df.dropna()

Clustering

In [6]:
le = LabelEncoder()

df["party_encoded"] = le.fit_transform(df["party"])
df["state_encoded"] = le.fit_transform(df["state"])
df["pc_name_encoded"] = le.fit_transform(df["pc_name"])
df["winning_candidate_encoded"] = le.fit_transform(df["winning_candidate"])


CREATE WIN/LOSS COLUMN

In [7]:
df["win"] = df["margin"].apply(lambda x: 1 if x > 0 else 0)

CREATE CLUSTERS (KMeans)

In [8]:
cluster_features = [
    "electors", "votes", "turnout", "margin",
    "party_encoded", "state_encoded"
]

df_numeric = df[cluster_features]

kmeans = KMeans(n_clusters=4, random_state=42)
df["cluster"] = kmeans.fit_predict(df_numeric)

CLASSIFICATION MODEL (Random Forest)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features for classification
features = [
    "electors", "votes", "turnout", "margin",
    "party_encoded", "state_encoded", "cluster"
]

X = df[features]
y = df["win"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           1       1.00      1.00      1.00       103

    accuracy                           1.00       103
   macro avg       1.00      1.00      1.00       103
weighted avg       1.00      1.00      1.00       103



EXPORTING FINAL CLEAN CSV

In [ ]:
df.to_csv("india_election_2019.csv", index=False)
print("CLEAN CSV CREATED SUCCESSFULLY!")


CLEAN CSV CREATED SUCCESSFULLY!
